Modified and verified by **Heejoon Moon**
- version: 11.28

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive'))
# 예: ['RWF-2000 Dataset', 'npy_dataset', ...]

In [ ]:
source_path = '/content/drive/MyDrive/컴퓨터비전개론'  # 실제 이름 맞춰서
print(os.listdir(source_path))          # ['train', 'val'] 나와야 함
print(os.listdir(source_path + '/train'))  # ['Fight', 'NonFight'] 등


## **0. Preprocessing Data**
- concat RGB frames and opticalflow
- transform videos to '.npy' format
  - '.npy' has 5 channels -> RGB (3) and opticalflow ((u, v) -> 2)
  - you can use another opticalflow algorithms

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

def getOpticalFlow(video):
    """Calculate dense optical flow of input video
    Args:
        video: the input video with shape of [frames,height,width,channel]. dtype=np.array
    Returns:
        flows_x: the optical flow at x-axis, with the shape of [frames,height,width,channel]
        flows_y: the optical flow at y-axis, with the shape of [frames,height,width,channel]
    """
    # initialize the list of optical flows
    gray_video = []
    for i in range(len(video)):
        img = cv2.cvtColor(video[i], cv2.COLOR_RGB2GRAY)
        gray_video.append(np.reshape(img, (224, 224, 1)))

    flows = []
    for i in range(0, len(video) - 1):
        # calculate optical flow between each pair of frames
        flow = cv2.calcOpticalFlowFarneback(gray_video[i], gray_video[i + 1], None, 0.5, 3, 15, 3, 5, 1.2,
                                            cv2.OPTFLOW_FARNEBACK_GAUSSIAN)
        # subtract the mean in order to eliminate the movement of camera
        flow[..., 0] -= np.mean(flow[..., 0])
        flow[..., 1] -= np.mean(flow[..., 1])
        # normalize each component in optical flow
        flow[..., 0] = cv2.normalize(flow[..., 0], None, 0, 255, cv2.NORM_MINMAX)
        flow[..., 1] = cv2.normalize(flow[..., 1], None, 0, 255, cv2.NORM_MINMAX)
        # Add into list
        flows.append(flow)

    # Padding the last frame as empty array
    flows.append(np.zeros((224, 224, 2)))

    return np.array(flows, dtype=np.float32)


def Video2Npy(file_path, resize=(224,224)):
    """Load video and tansfer it into .npy format
    Args:
        file_path: the path of video file
        resize: the target resolution of output video
    Returns:
        frames: gray-scale video
        flows: magnitude video of optical flows
    """
    # Load video
    cap = cv2.VideoCapture(file_path)
    # Get number of frames
    len_frames = int(cap.get(7))
    # Extract frames from video
    try:
        frames = []
        for i in range(len_frames-1):
            _, frame = cap.read()
            frame = cv2.resize(frame,resize, interpolation=cv2.INTER_AREA)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = np.reshape(frame, (224,224,3))
            frames.append(frame)
    except:
        print("Error: ", file_path, len_frames,i)
    finally:
        frames = np.array(frames)
        cap.release()

    # Get the optical flow of video
    flows = getOpticalFlow(frames)

    # Visualize optical flow map
    optical_flow_map = farneback_visual(flows)

    result = np.zeros((len(flows),224,224,5))
    result[...,:3] = frames
    result[...,3:] = flows

    return result

def farneback_visual(flows):
    # visualization farneback optical flow map
    # save the map as 'farneback_optical_flow.mp4'
    pass


def Save2Npy(file_dir, save_dir):   # modify this code to save the npy files for your directory or path
    """Transfer all the videos and save them into specified directory
    Args:
        file_dir: source folder of target videos
        save_dir: destination folder of output .npy files
    """
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    # List the files
    videos = os.listdir(file_dir)
    for v in tqdm(videos):
        # Split video name
        video_name = v.split('.')[0]
        # Get src
        video_path = os.path.join(file_dir, v)
        # Get dest
        save_path = os.path.join(save_dir, video_name+'.npy')
        # Load and preprocess video
        data = Video2Npy(file_path=video_path, resize=(224,224))
        data = np.uint8(data)
        # Save as .npy file
        np.save(save_path, data)

    return None

### convert data and save it (offline)

In [ ]:
source_path = '/content/drive/MyDrive/컴퓨터비전개론'
target_path = '/content/drive/MyDrive/컴퓨터비전개론'


for f1 in ['train', 'val']:
    for f2 in ['Fight', 'NonFight']:
        path1 = os.path.join(source_path, f1, f2)
        path2 = os.path.join(target_path, f1, f2)
        Save2Npy(file_dir=path1, save_dir=path2)

## **1. Build Data Loader**

In [ ]:
import torch
import torch.utils.data as data
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import cv2

class DataGenerator(Dataset):

    ### basic function templates for Dataset class in Pytorch : __init__, __len__, __getitem__###
    def __init__(self, directory, data_augmentation=True, phase='train'):
        self.phase=phase
        self.directory = directory
        self.data_aug = data_augmentation
        self.X_path, self.Y_dict = self.search_data()
        self.print_stats()

    def __len__(self):
        steps_per_epoch = int(len(self.X_path))
        return steps_per_epoch

    # get data and label corresponding to index using 'data_generation function'
    def __getitem__(self, index):
        data, label = self.data_generation(self.X_path[index])
        return data.float(), label

    #############################################################
    # -----------------codes down from this line are provided.------------------- #
    # call raw data using data path, and data processing including 'uniform sampling', 'color jittering', 'random flip', and 'normalize'
    #############################################################

    def load_data(self, path):
        data = np.load(path, mmap_mode='r') # Read the raw data from path
        data = self.uniform_sampling(data, target_frames=64) # Randomly sample number of target frames
        if self.data_aug: # If data is augmented...
            data[..., :3] = self.color_jitter(data[..., :3])
            data = self.random_flip(data, prob=0.5) # Random flip image into random direction
        data[..., :3] = self.normalize(data[..., :3]) # Normalize RGB
        data[..., 3:] = self.normalize(data[..., 3:]) # Normalize optical flows
        return data

    # shuffle data
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    # Normalize data
    def normalize(self, data):
        mean = data.mean()
        std = data.std()
        return (data - mean) / std

    def random_flip(self, video, prob):
        s = np.random.rand()
        if s < prob:
            video = np.flip(video, (2,)) # Flip in width direction
        return video

    def uniform_sampling(self, video, target_frames=64):
        # get total frames of input video and calculate sampling interval
        len_frames = int(len(video))
        interval = int(np.ceil(len_frames/target_frames))
        # init empty list for sampled video and
        sampled_video = []
        for i in range(0,len_frames,interval):
            sampled_video.append(video[i])
        # calculate numer of padded frames and fix it
        num_pad = target_frames - len(sampled_video) # num pad = # target frame - # current frame
        padding = []
        if num_pad>0:
            for i in range(-num_pad,0):
                try:
                    padding.append(video[i]) # Fill with the last video frame
                except:
                    padding.append(video[0])
            sampled_video += padding # Add padding results
        # get sampled video
        return np.array(sampled_video, dtype=np.float32)

    # Jitter = spread values
    def color_jitter(self, video):
        s_jitter = np.random.uniform(-0.2, 0.2)
        v_jitter = np.random.uniform(-30, 30)
        for i in range(len(video)):
            hsv = cv2.cvtColor(np.array(video[i]), cv2.COLOR_RGB2HSV) # Convert RGB -> HSV
            s = hsv[..., 1] + s_jitter # saturation jitter
            v = hsv[..., 2] + v_jitter # Value jitter
            # Flip
            s[s < 0] = 0
            s[s > 1] = 1
            v[v < 0] = 0
            v[v > 255] = 255
            hsv[..., 1] = s # set jittered saturation
            hsv[..., 2] = v # Set jittered value
            # just insert as numpy array since the progress is inefficient. tranform at the last #changes
            video[i] = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB) # Convert HSV -> RGB again
        return video

    # print current state sof the Dataset class
    def print_stats(self):
            self.n_files = len(self.X_path)
            self.n_classes = len(self.dirs)
            self.indexes = np.arange(len(self.X_path))
            np.random.shuffle(self.indexes)
            print("Found {} files belonging to {} classes.".format(self.n_files, self.n_classes))
            for i, label in enumerate(self.dirs):
                print('{:10s} : {}'.format(label, i))

    #############################################################
    # Fill the missing part of the code with functions given above
    #############################################################

    # get data and label(one-hot encoded) from path list/dictionary
    def search_data(self):
        X_path = []
        Y_dict = {}

        self.dirs = sorted(os.listdir(self.directory))  # Get sorted file directories
    # One-hot vector (class numbers = len(self.dirs))
        one_hots = np.eye(len(self.dirs), dtype=np.float32)   # ###Fill here###

        for i, folder in enumerate(self.dirs):
          folder_path = os.path.join(self.directory, folder)  # folder_path = .../train/Fight or .../train/NonFight
          for file in os.listdir(folder_path):
                if not file.endswith('.npy'):
                    continue
                file_path = os.path.join(folder_path, file)

            # Add file path into X_path
                X_path.append(file_path)                        # ###Fill here###

            # Assign one hot encoded vector into Y_dict
                Y_dict[file_path] = one_hots[i]                 # ###Fill here###

        return X_path, Y_dict

    # define batch x using 'load_data' function and batch_y from self.Y_dict.
    def data_generation(self, batch_path):
        video = self.load_data(batch_path)       # numpy array (T,H,W,5)
        label = self.Y_dict[batch_path]         # numpy array (num_classes,)

    # Since beacuause of flips..etc, stride can be negative num.
    # copy and make it as continuous array
    #flip 등 때문에 stride가 음수일 수 있으니, 복사해서 연속 배열로 만들기
        video = np.ascontiguousarray(video)
        label = np.ascontiguousarray(label)

        video = torch.from_numpy(video).float()
        label = torch.from_numpy(label).float()
        return video, label

## **2. Build Simple Model**
- this model is 'Flow Gated Network' proposed in 'RWF2000'
- you can use off-the-shelf architectures such as ResNet, EfficientNet, etc.
- model structure is produced in below image
- Fully-Connected layer is little bit different with image, so we provide Fully-Connected layer structure

![image.png](attachment:image.png)

In [ ]:
# the baseline model we used before using efficientnet
'''
import torch
import torch.nn as nn
import torch.nn.functional as F

class FusionModel(nn.Module):
    def __init__(self):
        super(FusionModel, self).__init__()
        self.relu=nn.ReLU(inplace=True)

        ## Hint: Please refer to above table for constructing layers
        # Construct block of RGB layers which takes RGB channel(3) as input
        self.rgb_block = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2)),     # 공간만 downsample

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(2, 2, 2)),     # 시공간 downsample
        )
        # Construct block of optical flow layers which takes the optical flow channel(2) as input
        self.opt_block = nn.Sequential(
            nn.Conv3d(2, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2)),

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(2, 2, 2)),
        )
        # Construct merging Block
        self.merge_block = nn.Sequential(
            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool3d((1, 1, 1))   # → (B, 128, 1, 1, 1)
        )
        # Fully Connected Layers
        self.fc1 = nn.Linear(128, 128)
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(128, 32)
        self.fc3 = nn.Linear(32, 2)

        # Initialize weights
        self.__init_weight()

    def forward(self, x):
        x = x.transpose(2,4)
        x = x.transpose(3,4)
        x = x.transpose(1,2)
        rgb = x[:,:3,:,:,:]
        opt = x[:,3:5,:,:,:]

        # Pass through the RGB data through the blocks of RGB layers
        rgb = self.rgb_block(rgb)
        # Pass through the optical flow data through the blocks of RGB layers
        opt = self.opt_block(opt)
        # Fuse by performing elementwise multiplication of rgb and opt tensors.
        fused = rgb * opt
        # Perform maxpooling of fused
        fused = F.max_pool3d(fused, kernel_size=2, stride=2)

        # Pass through the fused data into merging block
        merged = self.merge_block(fused)

        # Fully Connected Layers
        x = merged.view(merged.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)

        return x

    def __init_weight(self):
        for m in self.modules():
        # For 3D convolution layers - Kaiming Normal (He initialization)
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

        # For fully connected layers - Xavier initialization
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

        # For BatchNorm layers - set gamma=1, beta=0
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
'''

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class EfficientFusionModel(nn.Module):
    def __init__(self, num_classes=2):
        super(EfficientFusionModel, self).__init__()

        # -----------------------------------------------------------------
        # 1. RGB Stream (EfficientNet B0)
        # -----------------------------------------------------------------
        # call pretrained weights with ImageNet
        weights = models.EfficientNet_B0_Weights.DEFAULT
        self.rgb_backbone = models.efficientnet_b0(weights=weights)

        # 마지막 분류기(Classifier) 부분은 제거하고 특징 추출기(Feature Extractor)로만 사용합니다.
        # EfficientNet B0의 마지막 특징 맵 차원은 1280입니다.
        #get rid of the last classifier and just use as a feature extractor
        #last feature map dimension-> 1280
        self.rgb_backbone.classifier = nn.Identity()
        # -----------------------------------------------------------------
        # 2. Optical Flow Stream (EfficientNet B0)
        # -----------------------------------------------------------------
        self.opt_backbone = models.efficientnet_b0(weights=weights)
        self.opt_backbone.classifier = nn.Identity()

        # modify the first conv as 2channel layer and just reconnect to layers after that
        # Conv2d(3, 32, ...) ->  Conv2d(2, 32, ...)
        first_conv = self.opt_backbone.features[0][0]
        new_first_conv = nn.Conv2d(
            in_channels=2,
            out_channels=first_conv.out_channels,
            kernel_size=first_conv.kernel_size,
            stride=first_conv.stride,
            padding=first_conv.padding,
            bias=False
        )

        #accordingly modify the original 3 channel weights and initialize as 2channel
        #--> transfer learning
        with torch.no_grad():
            new_first_conv.weight[:] = torch.mean(first_conv.weight, dim=1, keepdim=True)

        self.opt_backbone.features[0][0] = new_first_conv

        # -----------------------------------------------------------------
        # 3. Fusion & Classification Block
        # -----------------------------------------------------------------
        # add 2560=RGB(1280) + Flow(1280) dimensions feature
        self.fusion_fc = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280 * 2, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # x Input Shape : (Batch, Time, Height, Width, Channel=5)
        if x.dim() == 5 and x.shape[-1] == 5:
            x = x.permute(0, 4, 1, 2, 3) # (B, 5, T, H, W)

        b, c, t, h, w = x.size()

        # seperate channels
        rgb = x[:, :3, :, :, :]  # (B, 3, T, H, W)
        opt = x[:, 3:5, :, :, :] # (B, 2, T, H, W)

        # -----------------------------------------------------
        #since efficientnet is for 2d images, group the (Batch * Time) as a batch
        # Shape : (B, C, T, H, W) -> (B*T, C, H, W)
        # -----------------------------------------------------
        rgb = rgb.permute(0, 2, 1, 3, 4).reshape(b * t, 3, h, w)
        opt = opt.permute(0, 2, 1, 3, 4).reshape(b * t, 2, h, w)

        # Feature Extraction
        # Output Shape: (B*T, 1280)
        rgb_feat = self.rgb_backbone(rgb)
        opt_feat = self.opt_backbone(opt)

        # restore dimensions (B, T, 1280)
        rgb_feat = rgb_feat.view(b, t, -1)
        opt_feat = opt_feat.view(b, t, -1)

        # -----------------------------------------------------
        # Temporal Pooling: evaluate average of time axis
        # -----------------------------------------------------
        rgb_feat = torch.mean(rgb_feat, dim=1) # (B, 1280)
        opt_feat = torch.mean(opt_feat, dim=1) # (B, 1280)

        # Fusion (Concatenation)
        fused = torch.cat((rgb_feat, opt_feat), dim=1) # (B, 2560)

        # Final Classification
        out = self.fusion_fc(fused)

        return out

## **3. Training the Model**
- set hyper-parameters for training

In [ ]:
!pip install wandb

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from tqdm import tqdm
import wandb


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# define model, optimizer and criterion
model = EfficientFusionModel().to(device)                          # ← model
optimizer = optim.SGD(                                    # ←optimizer
    model.parameters(),
    lr=0.003,
    weight_decay=1e-6,
    momentum=0.9,
    nesterov=True,
)
loss_fn = nn.CrossEntropyLoss()                           # ← loss func

trainset_path = '/train'
validation_path = '/val'
#scheduler = StepLR(optimizer, step_size=10, gamma=0.7)

trainset_path = '/content/drive/MyDrive/컴퓨터비전개론/train'
validation_path = '/content/drive/MyDrive/컴퓨터비전개론/val'
# define dataset and dataloader
train_dataset =DataGenerator(trainset_path,  phase='train', data_augmentation=True)
val_dataset =DataGenerator(validation_path, phase='val',   data_augmentation=False)
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# epoch = 30
# learning rate = 0.003

# optimizer = SGD
# weight decay = 1e-6
# momentum = 0.9
# nesterov = True
# gamma = 0.7

# loss = CrossEntropy

# assign device cpu or gpu
min_loss = np.inf


In [ ]:
def _train(self):
    model.train()
    acc_temp = 0
    running_loss = 0

    # forward propagation and backpropagation
    # calculate accuracy and loss on training set

    train_acc = acc_temp / len(train_loader.dataset)
    train_loss = running_loss / len(train_loader.dataset)

    return train_acc, train_loss

In [ ]:
def _val(self):
    model.eval()
    with torch.no_grad():
        running_loss_val = 0
        acc_temp_val =0

        # calculate accuracy and loss on validation set

        val_acc = acc_temp_val / len(val_loader.dataset)
        val_loss = running_loss_val / len(val_loader.dataset)

    return val_acc, val_loss

In [ ]:
import random
import numpy as np
import os
import torch
import wandb

#import madatory libraries for mixed precision
from torch.cuda.amp import autocast, GradScaler

seed = 0
random.seed(seed)
np.random.seed(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(seed)
if device == 'cuda':
    print('gpu device is using')
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
num_epochs = 30

#  min_loss initialization
min_loss = float('inf')

#  GradScaler initialization
scaler = GradScaler()

wandb.login()
wandb.init(project='computer vision', name='컴퓨터비전개론')

for epoch in range(num_epochs):
    print(f"Epoch [{epoch+1}/{num_epochs}] 시작!")

    train_loss = 0.0
    val_loss = 0.0
    train_correct = 0
    train_total = 0
    val_correct = 0
    val_total = 0


    # 1. Training Loop (Mixed Precision )
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        # wrap with autocast and perform Forward (save memory)
        with autocast():
            pred = model(x)
            loss = loss_fn(pred, y)

        #backpropagation and update weights with scaler
        #  loss.backward() -> optimizer.step()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        #scheduler.step() <-- lessen the auroc score, dont use

        train_loss += loss.item()

        if y.dim() > 1 and y.size(1) > 1:
            y_labels = torch.argmax(y, dim=1)
        else:
            y_labels = y

        _, predicted = torch.max(pred.data, 1)
        train_total += y_labels.size(0)
        train_correct += (predicted == y_labels).sum().item()

    # ==========================================
    # 2. Validation Loop
    # ==========================================
    model.eval()
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            loss = loss_fn(pred, y)
            val_loss += loss.item()
            if y.dim() > 1 and y.size(1) > 1:
                y_labels = torch.argmax(y, dim=1)
            else:
                y_labels = y

            _, predicted = torch.max(pred.data, 1)
            val_total += y_labels.size(0)
            val_correct += (predicted == y_labels).sum().item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    # Log results
    avg_train_loss = train_loss / len(train_loader)
    avg_train_acc = 100 * train_correct / train_total

    avg_val_loss = val_loss / len(val_loader)
    avg_val_acc = 100 * val_correct / val_total

    print(f"Epoch [{epoch+1}] Train Acc: {avg_train_acc:.2f}% | Val Acc: {avg_val_acc:.2f}%")
    wandb.log({
        "train_loss": avg_train_loss,
        "train_acc": avg_train_acc,
        "val_loss": avg_val_loss,
        "val_acc": avg_val_acc,
        "epoch": epoch + 1
    })

    save_dir = '/content/drive/MyDrive/컴퓨터비전개론/models'
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # Save the best model
    if avg_val_loss < min_loss:
        min_loss = avg_val_loss
        save_path = os.path.join(save_dir, f'FusionModel_best.pth')
        torch.save(model.state_dict(), save_path)
        print(f"Best model updated at epoch {epoch+1} (Loss: {min_loss:.4f})")

In [ ]:
from torch.utils.data import DataLoader




best_model = '/content/drive/MyDrive/컴퓨터비전개론/models/FusionModel_best.pth'
test_path = '/content/drive/MyDrive/컴퓨터비전개론/val'

test_dataset = DataGenerator(directory=test_path, data_augmentation=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True, num_workers=0)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

model.load_state_dict(torch.load(best_model, map_location=device))
model.eval()

y_true = []
y_score = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        pred = model(x)
        score = torch.softmax(pred, dim=1)[:,1].cpu().numpy()
        y_score.extend(score)
        y_true.extend(y.numpy())

# calculate accuracy
pred_labels = (np.array(y_score) > 0.5).astype(int)
true_labels = np.argmax(np.array(y_true), axis=1)  # One-Hot -> Index
accuracy = (pred_labels == true_labels).mean()

print(f'Accuracy: {accuracy:.4f}')

print(f'Accuracy: {accuracy:.4f}')

# calculate AUROC
y_true_index = np.argmax(np.array(y_true), axis=1)

auc = roc_auc_score(y_true_index, y_score)
print(f'AUROC: {auc:.4f}')

fpr, tpr, _ = roc_curve(y_true_index, y_score)
print(f'AUROC: {auc:.4f}')

# plot AUROC curve
fpr, tpr, _ = roc_curve(y_true_index, y_score)
plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()
